In [71]:
import pandas as pd
import re
import ast

pd.options.display.max_columns = None

In [72]:
dds = pd.read_excel("./original_dds.xlsx")
ho = pd.read_excel("./health_office.xlsx")
icd = pd.read_excel("./icd10.xlsx")

In [73]:
before = len(dds)

dds = dds[
    ~dds["ชื่อ"].astype(str).str.contains(
        "ทดสอบ",
        case=False,
        na=False
    )
]

after = len(dds)

print("ก่อนลบ:", before)
print("หลังลบ:", after)
print("ลบไป:", before - after, "แถว")

ก่อนลบ: 2171
หลังลบ: 2121
ลบไป: 50 แถว


In [74]:
dds = dds .drop(columns=[
    "รหัสหน่วยงาน", "หน่วยงาน", "CID",  "วันที่ส่งรายงาน", "คำนำหน้า", "ชื่อ", "นามสกุล", "เพศ", "อายุปี", "อายุเดือน",
    "สัญชาติ", "อาชีพ", "เบอร์โทรศัพท์", "จังหวัดขณะป่วย", "ที่อยู่ปัจจุบัน", "หมู่(ที่อยู่ปัจจุบัน)", "ถนน(ที่อยู่ปัจจุบัน)",
    "ที่อยู่ขณะป่วย", "หมู่ขณะป่วย", "ถนนขณะป่วย", "รหัสจังหวัด(ที่อยู่ปัจจุบัน)", "รหัสอำเภอ(ที่อยู่ปัจจุบัน)",
    "รหัสตำบล(ที่อยู่ปัจจุบัน)", 
    "อำเภอขณะป่วย", "ตำบลขณะป่วย", 
    "วันที่เริ่มมีอาการ", "วันที่วินิจฉัยโรค", "organism",
    "ประเภทผู้ป่วย", "ความรุนแรง", "ใส่เครื่องช่วยหายใจ", "รหัสสภาพผู้ป่วย", "สภาพผู้ป่วย", "รหัสกลุ่มโรค",
    "รหัสวิธีการตรวจ Lab", "วิธีการตรวจ Lab", "วันที่รายงานผล Lab", "ผล Lab", "วันที่เก็บตัวอย่าง", "รหัส LAB(HIS)", 
    "ชื่อรายการ Lab(HIS)", "รหัส TMLT", "วันที่เสียชีวิต", "สาเหตุการเสียชีวิต", "status", "สถานะ", "หมายเหตุ",
    "วันที่อนุมัติรายงาน", "วันที่ Update", "complication", "รหัสจังหวัดที่รับรักษา"
])

In [75]:
dds = dds.rename(columns={
    "วันที่เริ่มรักษา": "date",
    "โรงพยาบาลที่กำลังรักษา": "hospcode",
    "Diagnosis ICD10": "icd10",
    "diagnosis_icd10_list": "icd10_list"
})

In [76]:
dds = dds[dds["icd10"].str.contains("Z581", na=False)]
print("Shape after filter:", dds.shape)

Shape after filter: (2120, 4)


In [77]:
# clean icd10 (ค่าเดียว)
def clean_icd(val):
    if pd.isna(val):
        return val
    val = str(val)
    # ลบทุกอย่างที่ไม่ใช่ A-Z a-z 0-9
    val = re.sub(r'[^A-Za-z0-9]', '', val)
    # แปลงเป็นตัวใหญ่
    val = val.upper()
    return val


# clean icd10_list (หลายค่า เช่น 'J40','Z581')
def clean_icd_list(val):
    if pd.isna(val):
        return val
    val = str(val)
    # ลบ ' ออก
    val = val.replace("'", "")
    # แยกค่าด้วย comma
    items = val.split(",")

    cleaned = []
    for item in items:
        item = re.sub(r'[^A-Za-z0-9]', '', item)
        item = item.upper()

        if item:
            cleaned.append(item)

    # รวมกลับเป็น comma
    return ",".join(cleaned)


# ใช้กับ DataFrame dds
if "icd10" in dds.columns:
    dds["icd10"] = dds["icd10"].apply(clean_icd)

if "icd10_list" in dds.columns:
    dds["icd10_list"] = dds["icd10_list"].apply(clean_icd_list)

In [78]:
def clean_code(x):
    if pd.isna(x):
        return pd.NA
    
    x = str(x).strip()

    # กรณี Excel อ่านเป็น 12345.0
    if x.endswith(".0"):
        x = x[:-2]

    return x

dds["hospcode_key"] = dds["hospcode"].apply(clean_code)

for col in ["hospcode9", "hospcode9old", "hospcode"]:
    if col in ho.columns:
        ho[col] = ho[col].apply(clean_code)

# -------------------------
# 2) คอลัมน์ที่ต้องการดึงจาก ho
# -------------------------
ho_cols = [
    "hospcode_name",
    "county",
    "province_id",
    "province_name",
    "district_name",
    "subdistrict_name"
]

# -------------------------
# 3) สร้าง lookup ตามลำดับความสำคัญ
# -------------------------
lookup_hospcode9 = (
    ho.dropna(subset=["hospcode9"])
      .drop_duplicates(subset=["hospcode9"])
      .set_index("hospcode9")[ho_cols]
      .to_dict("index")
)

lookup_hospcode9old = (
    ho.dropna(subset=["hospcode9old"])
      .drop_duplicates(subset=["hospcode9old"])
      .set_index("hospcode9old")[ho_cols]
      .to_dict("index")
)

lookup_hospcode = (
    ho.dropna(subset=["hospcode"])
      .drop_duplicates(subset=["hospcode"])
      .set_index("hospcode")[ho_cols]
      .to_dict("index")
)

# -------------------------
# 4) ฟังก์ชันเช็คข้อมูลตามลำดับ
# -------------------------
def match_hospital(code):
    if pd.isna(code):
        return pd.Series({
            "hospcode_name": "ไม่ทราบ",
            "county": "ไม่ทราบ",
            "province_id": "ไม่ทราบ",
            "province_name": "ไม่ทราบ",
            "district_name": "ไม่พบ",
            "subdistrict_name": "ไม่พบ",
            "match_from": "ไม่พบ"
        })

    if code in lookup_hospcode9:
        data = lookup_hospcode9[code].copy()
        data["match_from"] = "hospcode9"
        return pd.Series(data)

    if code in lookup_hospcode9old:
        data = lookup_hospcode9old[code].copy()
        data["match_from"] = "hospcode9old"
        return pd.Series(data)

    if code in lookup_hospcode:
        data = lookup_hospcode[code].copy()
        data["match_from"] = "hospcode"
        return pd.Series(data)

    return pd.Series({
        "hospcode_name": "ไม่ทราบ",
        "county": "ไม่ทราบ",
        "province_id": "ไม่ทราบ",
        "province_name": "ไม่ทราบ",
        "district_name": "ไม่พบ",
        "subdistrict_name": "ไม่พบ",
        "match_from": "ไม่พบ"
    })

# -------------------------
# 5) join ข้อมูลเข้า dds
# -------------------------
matched = dds["hospcode_key"].apply(match_hospital)

dds = pd.concat([dds, matched], axis=1)

# ถ้าไม่ต้องการคอลัมน์ช่วย
dds = dds.drop(columns=["hospcode_key"])

In [79]:
# แปลง date เป็น datetime (กัน error ถ้าเป็น string)
dds["date"] = pd.to_datetime(dds["date"], errors="coerce")
# ดึง year
dds["year"] = dds["date"].dt.year
# ดึง week (ISO week)
dds["week"] = dds["date"].dt.isocalendar().week
# ดึง month
dds["month"] = dds["date"].dt.month

In [80]:
# diagnosis ที่ต้องใช้ตรวจ
diagnosis_prefix = {"J44", "J45", "I21", "I22", "I24", "H10", "L30", "L50"}

# exception ที่ต้องคงไว้
exceptions = {"J442", "L309"}

def clean_icd10_list(val):
    if pd.isna(val):
        return val

    val = str(val).upper()

    # แยกหลายค่า เช่น J440,J441,L309
    items = re.split(r'[,\s]+', val)

    cleaned = []

    for item in items:
        if not item:
            continue

        # ลบตัวแปลก ยกเว้น .
        item = re.sub(r'[^A-Z0-9]', '', item)

        # ถ้าเป็น exception ให้คงไว้
        if item in exceptions:
            cleaned.append(item)
            continue

        # เอา prefix 3 ตัวแรก
        prefix = item[:3]

        # ถ้า prefix อยู่ใน diagnosis → ตัดเหลือ 3 ตัว
        if prefix in diagnosis_prefix:
            cleaned.append(prefix)
        else:
            cleaned.append(item)

    return ",".join(cleaned)


# ใช้กับ dds
dds["icd10_list"] = dds["icd10_list"].apply(clean_icd10_list)

print(dds["icd10_list"].head())

0                      J302,J47,Z581
1                      J310,Z581,Y97
2    E119,C189,E789,H10,I10,Z581,Y97
3                     E050,L309,Z581
4                       Z518,L50,Y97
Name: icd10_list, dtype: object


In [81]:
# ลบคอลัมน์
dds = dds.drop(columns=["match_from", "date"], errors="ignore")

# เรียงคอลัมน์ใหม่
new_order = [
    "year",
    "week",
    "month",
    "hospcode",
    "hospcode_name",
    "county",
    "province_id",
    "province_name",
    "district_name",
    "subdistrict_name",
    "icd10",
    "icd10_list",
    # "diagnosis_type"
]

# เรียงคอลัมน์
dds = dds[new_order]

In [82]:
# print(dds.columns)

In [83]:
codes = (
    dds["icd10_list"]
    .dropna()
    .astype(str)
    .str.upper()
    .str.split(",")
)

# flatten
all_codes = codes.explode()

# clean ตัวแปลก
all_codes = all_codes.str.replace(r"[^A-Z0-9]", "", regex=True)

# ลบค่าว่าง
all_codes = all_codes[all_codes != ""]

# -------------------------
# 2) นับจำนวนแต่ละรหัส
# -------------------------
code_counts = (
    all_codes
    .value_counts()
    .reset_index()
)

code_counts.columns = ["icd10_code", "count"]

# -------------------------
# 3) เพิ่ม group ตัวแรก
# -------------------------
code_counts["group"] = code_counts["icd10_code"].str[0]

# -------------------------
# 4) เรียง A-Z แล้ว 0-9
# -------------------------
# กำหนดลำดับตัวอักษร
letters = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")
numbers = list("0123456789")

order = letters + numbers

code_counts["group"] = pd.Categorical(
    code_counts["group"],
    categories=order,
    ordered=True
)

# sort
code_counts = code_counts.sort_values(
    by=["group", "icd10_code"]
)

# -------------------------
# 5) groupby ตัวแรก (สรุปยอดรวม)
# -------------------------
group_summary = (
    code_counts
    .groupby("group", observed=True)["count"]
    .sum()
    .reset_index()
)

# -------------------------
# 6) export CSV
# -------------------------
code_counts.to_csv(
    "icd10_code_counts_sorted.csv",
    index=False,
    encoding="utf-8-sig"
)

group_summary.to_csv(
    "icd10_group_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Export completed:")
print("- icd10_code_counts_sorted.csv")
print("- icd10_group_summary.csv")

Export completed:
- icd10_code_counts_sorted.csv
- icd10_group_summary.csv


In [84]:
dds.head()

,year,week,month,hospcode,hospcode_name,county,province_id,province_name,district_name,subdistrict_name,icd10,icd10_list
0,2026,17,4,10674,โรงพยาบาลเชียงรายประชานุเคราะห์,1,57,เชียงราย,เมืองเชียงราย,เวียง,Z581,"J302,J47,Z581"
1,2026,17,4,10674,โรงพยาบาลเชียงรายประชานุเคราะห์,1,57,เชียงราย,เมืองเชียงราย,เวียง,Z581,"J310,Z581,Y97"
2,2026,17,4,10674,โรงพยาบาลเชียงรายประชานุเคราะห์,1,57,เชียงราย,เมืองเชียงราย,เวียง,Z581,"E119,C189,E789,H10,I10,Z581,Y97"
3,2026,17,4,10674,โรงพยาบาลเชียงรายประชานุเคราะห์,1,57,เชียงราย,เมืองเชียงราย,เวียง,Z581,"E050,L309,Z581"
4,2026,17,4,4858,โรงพยาบาลส่งเสริมสุขภาพตำบลหนองปลาปาก,8,43,หนองคาย,ศรีเชียงใหม่,หนองปลาปาก,Z581,"Z518,L50,Y97"


In [85]:
def clean_z581(x):
    if pd.isna(x):
        return pd.NA

    # แปลง string -> list
    if isinstance(x, str):
        x = x.strip()

        # กันค่าว่าง
        if x in ["", "[]", "['']", '[""]']:
            return pd.NA

        try:
            codes = ast.literal_eval(x)
        except:
            x = x.strip("[]")
            codes = [
                i.strip().replace("'", "").replace('"', '')
                for i in x.split(",")
                if i.strip()
            ]
    else:
        codes = list(x)

    # ลบค่าว่าง '', None
    codes = [
        str(c).strip().upper()
        for c in codes
        if pd.notna(c) and str(c).strip() != ""
    ]

    # ถ้าไม่มีค่าเลย
    if len(codes) == 0:
        return pd.NA

    # ถ้ามีแค่ Z581 ตัวเดียว → เก็บไว้
    if len(codes) == 1 and codes[0] == "Z581":
        return codes

    # ถ้ามีหลายตัว → ลบ Z581
    cleaned = [c for c in codes if c != "Z581"]

    # ถ้าลบแล้วไม่เหลือค่า
    if len(cleaned) == 0:
        return pd.NA

    return cleaned


# ใช้งาน
dds["icd10_list"] = dds["icd10_list"].apply(clean_z581)

In [86]:
dds["icd10_list"] = (
    dds["icd10_list"]
    .astype(str)
    .str.replace(r"[\[\]']", "", regex=True)  # ลบ [ ] ' ,
    .str.replace(r"\s+", " ", regex=True)      # ลบช่องว่างเกิน
    .str.strip()
)

In [87]:
dds.head()

,year,week,month,hospcode,hospcode_name,county,province_id,province_name,district_name,subdistrict_name,icd10,icd10_list
0,2026,17,4,10674,โรงพยาบาลเชียงรายประชานุเคราะห์,1,57,เชียงราย,เมืองเชียงราย,เวียง,Z581,"J302, J47"
1,2026,17,4,10674,โรงพยาบาลเชียงรายประชานุเคราะห์,1,57,เชียงราย,เมืองเชียงราย,เวียง,Z581,"J310, Y97"
2,2026,17,4,10674,โรงพยาบาลเชียงรายประชานุเคราะห์,1,57,เชียงราย,เมืองเชียงราย,เวียง,Z581,"E119, C189, E789, H10, I10, Y97"
3,2026,17,4,10674,โรงพยาบาลเชียงรายประชานุเคราะห์,1,57,เชียงราย,เมืองเชียงราย,เวียง,Z581,"E050, L309"
4,2026,17,4,4858,โรงพยาบาลส่งเสริมสุขภาพตำบลหนองปลาปาก,8,43,หนองคาย,ศรีเชียงใหม่,หนองปลาปาก,Z581,"Z518, L50, Y97"


In [88]:
dds = dds.copy()
icd = icd.copy()

# เพิ่ม id คน/เคส
dds.insert(0, "person_id", range(1, len(dds) + 1))

# เตรียม icd10_list
dds["icd10_list"] = dds["icd10_list"].fillna("").astype(str)

# -------------------------
# เตรียมตาราง icd
# -------------------------
icd["sub_code_clean"] = (
    icd["sub-code"]
    .fillna("")
    .astype(str)
    .str.upper()
    .str.replace(".", "", regex=False)
    .str.strip()
)

# สำหรับ Disease Type ใช้แค่ 3 ตัวแรก
icd["icd3"] = icd["sub_code_clean"].str[:3]

# lookup Disease Type จาก 3 ตัวแรก
icd_type_lookup = (
    icd.dropna(subset=["icd3"])
       .drop_duplicates(subset=["icd3"])
       [["icd3", "Disease Type"]]
)

# lookup disease จากรหัสเต็ม
icd_disease_lookup = (
    icd.dropna(subset=["sub_code_clean"])
       .drop_duplicates(subset=["sub_code_clean"])
       [["sub_code_clean", "disease"]]
)

# -------------------------
# แตก icd10_list เป็นหลายแถว
# -------------------------
dds_exploded = (
    dds.assign(
        icd10_code=dds["icd10_list"]
        .str.upper()
        .str.replace(".", "", regex=False)
        .str.split(",")
    )
    .explode("icd10_code")
)

dds_exploded["icd10_code"] = dds_exploded["icd10_code"].str.strip()
dds_exploded["icd3"] = dds_exploded["icd10_code"].str[:3]

# -------------------------
# merge Disease Type ด้วย 3 ตัวแรก
# -------------------------
dds_merge = dds_exploded.merge(
    icd_type_lookup,
    on="icd3",
    how="left"
)

# -------------------------
# merge disease ด้วยรหัสเต็มทุกตัว
# -------------------------
dds_merge = dds_merge.merge(
    icd_disease_lookup,
    left_on="icd10_code",
    right_on="sub_code_clean",
    how="left"
)

# ลบคอลัมน์ช่วย
dds_merge = dds_merge.drop(columns=["icd3", "sub_code_clean"], errors="ignore")

print(dds_merge[["icd10_code", "Disease Type", "disease"]].head())

  icd10_code                               Disease Type  \
0       J302                        โรคระบบทางเดินหายใจ   
1        J47                        โรคระบบทางเดินหายใจ   
2       J310                        โรคระบบทางเดินหายใจ   
3        Y97  สาเหตุภายนอกของการเจ็บป่วยและการเสียชีวิต   
4       E119  โรคระบบต่อมไร้ท่อ โภชนาการ และเมแทบอลิซึม   

                                     disease  
0           Other seasonal allergic rhinitis  
1                             Bronchiectasis  
2                           Chronic rhinitis  
3  Environmental-pollution-related condition  
4                                        NaN  


In [89]:
dds_merge.head(10)

,person_id,year,week,month,hospcode,hospcode_name,county,province_id,province_name,district_name,subdistrict_name,icd10,icd10_list,icd10_code,Disease Type,disease
0,1,2026,17,4,10674,โรงพยาบาลเชียงรายประชานุเคราะห์,1,57,เชียงราย,เมืองเชียงราย,เวียง,Z581,"J302, J47",J302,โรคระบบทางเดินหายใจ,Other seasonal allergic rhinitis
1,1,2026,17,4,10674,โรงพยาบาลเชียงรายประชานุเคราะห์,1,57,เชียงราย,เมืองเชียงราย,เวียง,Z581,"J302, J47",J47,โรคระบบทางเดินหายใจ,Bronchiectasis
2,2,2026,17,4,10674,โรงพยาบาลเชียงรายประชานุเคราะห์,1,57,เชียงราย,เมืองเชียงราย,เวียง,Z581,"J310, Y97",J310,โรคระบบทางเดินหายใจ,Chronic rhinitis
3,2,2026,17,4,10674,โรงพยาบาลเชียงรายประชานุเคราะห์,1,57,เชียงราย,เมืองเชียงราย,เวียง,Z581,"J310, Y97",Y97,สาเหตุภายนอกของการเจ็บป่วยและการเสียชีวิต,Environmental-pollution-related condition
4,3,2026,17,4,10674,โรงพยาบาลเชียงรายประชานุเคราะห์,1,57,เชียงราย,เมืองเชียงราย,เวียง,Z581,"E119, C189, E789, H10, I10, Y97",E119,โรคระบบต่อมไร้ท่อ โภชนาการ และเมแทบอลิซึม,NaN
5,3,2026,17,4,10674,โรงพยาบาลเชียงรายประชานุเคราะห์,1,57,เชียงราย,เมืองเชียงราย,เวียง,Z581,"E119, C189, E789, H10, I10, Y97",C189,เนื้องอก (รวมถึงมะเร็ง),"Colon, unspecified"
6,3,2026,17,4,10674,โรงพยาบาลเชียงรายประชานุเคราะห์,1,57,เชียงราย,เมืองเชียงราย,เวียง,Z581,"E119, C189, E789, H10, I10, Y97",E789,โรคระบบต่อมไร้ท่อ โภชนาการ และเมแทบอลิซึม,"Disorder of lipoprotein metabolism, unspecified"
7,3,2026,17,4,10674,โรงพยาบาลเชียงรายประชานุเคราะห์,1,57,เชียงราย,เมืองเชียงราย,เวียง,Z581,"E119, C189, E789, H10, I10, Y97",H10,โรคตารวมส่วนประกอบของตา,Conjunctivitis
8,3,2026,17,4,10674,โรงพยาบาลเชียงรายประชานุเคราะห์,1,57,เชียงราย,เมืองเชียงราย,เวียง,Z581,"E119, C189, E789, H10, I10, Y97",I10,โรคระบบไหลเวียนเลือด,Essential (primary) hypertension
9,3,2026,17,4,10674,โรงพยาบาลเชียงรายประชานุเคราะห์,1,57,เชียงราย,เมืองเชียงราย,เวียง,Z581,"E119, C189, E789, H10, I10, Y97",Y97,สาเหตุภายนอกของการเจ็บป่วยและการเสียชีวิต,Environmental-pollution-related condition


In [90]:
print("ก่อนลบ:", len(dds_merge))

before = len(dds_merge)

# เติมค่าว่างเป็น Z581
dds_merge["icd10_code"] = (
    dds_merge["icd10_code"]
    .fillna("Z581")
    .astype(str)
    .str.strip()
    .replace("", "Z581")
)

# เตรียมคอลัมน์ clean
code_clean = (
    dds_merge["icd10_code"]
    .astype(str)
    .str.upper()
    .str.replace(".", "", regex=False)
    .str.strip()
)

# ลบเฉพาะตัวเลขล้วน
cond_numeric = code_clean.str.fullmatch(r"\d+")

dds_merge = dds_merge[~cond_numeric]

after = len(dds_merge)

print("หลังลบ:", after)
print("ลบไป:", before - after, "แถว")

ก่อนลบ: 5804
หลังลบ: 5139
ลบไป: 665 แถว


In [91]:
dds_merge["icd10_old"] = dds_merge["icd10_code"]

In [92]:
dds_merge.head()

,person_id,year,week,month,hospcode,hospcode_name,county,province_id,province_name,district_name,subdistrict_name,icd10,icd10_list,icd10_code,Disease Type,disease,icd10_old
0,1,2026,17,4,10674,โรงพยาบาลเชียงรายประชานุเคราะห์,1,57,เชียงราย,เมืองเชียงราย,เวียง,Z581,"J302, J47",J302,โรคระบบทางเดินหายใจ,Other seasonal allergic rhinitis,J302
1,1,2026,17,4,10674,โรงพยาบาลเชียงรายประชานุเคราะห์,1,57,เชียงราย,เมืองเชียงราย,เวียง,Z581,"J302, J47",J47,โรคระบบทางเดินหายใจ,Bronchiectasis,J47
2,2,2026,17,4,10674,โรงพยาบาลเชียงรายประชานุเคราะห์,1,57,เชียงราย,เมืองเชียงราย,เวียง,Z581,"J310, Y97",J310,โรคระบบทางเดินหายใจ,Chronic rhinitis,J310
3,2,2026,17,4,10674,โรงพยาบาลเชียงรายประชานุเคราะห์,1,57,เชียงราย,เมืองเชียงราย,เวียง,Z581,"J310, Y97",Y97,สาเหตุภายนอกของการเจ็บป่วยและการเสียชีวิต,Environmental-pollution-related condition,Y97
4,3,2026,17,4,10674,โรงพยาบาลเชียงรายประชานุเคราะห์,1,57,เชียงราย,เมืองเชียงราย,เวียง,Z581,"E119, C189, E789, H10, I10, Y97",E119,โรคระบบต่อมไร้ท่อ โภชนาการ และเมแทบอลิซึม,NaN,E119


In [93]:
# ทำความสะอาดและให้เหลือ 3 หลักแรกของ ICD-10
dds_merge["icd10_code"] = (
    dds_merge["icd10_code"]
    .astype(str)
    .str.upper()
    .str.strip()
    .str.replace(".", "", regex=False)  # เอาจุดออก เช่น J44.2 -> J442
    .str.extract(r"([A-Z]\d{2})", expand=False)
)

In [94]:
dds_merge = dds_merge.drop(columns=[
    "hospcode", "hospcode_name", "province_id", "icd10", "icd10_list", 
])

In [95]:
dds_merge.to_csv("dashboard_dds.csv", index=False, encoding="utf-8-sig")